# Generic search: significance maps

The corrected map of the generic search (`residual`: density − background) is in stars per pixel. Its noise grows with
the number of stars a match filter selects: a wider filter, another VIS photometry or isochrone model gives a noisier
residual map, even when it finds a stream with a higher SNR. The maps of two match filters are then not comparable by
eye.

The significance (`generic_search_significance`) divides the residual by its Poisson noise, sqrt(background × Σk²),
k the Gaussian smoothing kernel on the pixels: 1 is one standard deviation of the noise, whatever the number of stars.
Its standard deviation over the footprint is 1 for pure Poisson noise; above 1, the rest: real structures, and the error
of the polynomial background (about 5% of the density on a uniform mock, whatever the density, so that it dominates
the Poisson noise where the density is high).

Two selections, on the same stars (morphology cuts, `POINT_LIKE_PROB` ≥ 0.5, Planck dust correction, the current
default):
- **previous**: Marigo2017, PSF VIS, the streamobs width and red cap (the generic search before 2026-11-18);
- **current defaults**: `DEFAULT_ISOCHRONE_MODEL` and `DEFAULT_MATCH_FILTER` (Dotter2016, template-fit VIS,
  `error_multiplier` 3, `red_cap_value` 0.55; `notebooks/optimization/stream_optimization.ipynb`).

In [ ]:
%load_ext autoreload
%autoreload 2

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from streameucliddes.analysis.cmd_match_filter import get_match_filter
from streameucliddes.analysis.generic_stream_search import (
    generic_search_significance,
    generic_search_single_dm,
    prepare_generic_search,
)
from streameucliddes.catalogs.euclid_catalog import default_star_cuts, get_euclid_stars
from streameucliddes.catalogs.survey_footprint import survey_footprint_healpix_mask
from streameucliddes.plotting.plot_generic_search import plot_generic_search_residual_map
from streameucliddes.unit_conversions import get_isochrone_model

## 1. Stars, footprint and the two searches

The stars are read once, with the template-fit VIS (`vis_photometry="templfit"`, the default: the PSF VIS is read
too), and shared by the two searches (neither copies the catalog).

In [ ]:
CATALOG = "euclid_dr1"
SUBSET = "south"  # "south", "northwest", "northeast" or None
NSIDE = 512  # pixels of about 0.11 deg
SMOOTHING_DEG = 0.3
DISTANCE_MODULI = [16.0, 17.0, 18.0, 19.0]
SIGNIFICANCE_LIMITS = (-4.0, 4.0)  # colour scale of the significance maps, the same for every selection

# The two selections: match filter and isochrone model (None: DEFAULT_ISOCHRONE_MODEL)
SELECTIONS = {
    "previous": {
        "match_filter": {"name": "euclid_vis_h_previous", "reference_band": ("euclid_dr1", "VIS"),
                         "color_bands": [("euclid_dr1", "H")], "vis_photometry": "psf"},
        "isochrone_model": "Marigo2017",
    },
    "current defaults": {"match_filter": get_match_filter(), "isochrone_model": None},
}
for name, selection in SELECTIONS.items():
    print(f"{name}: {get_isochrone_model(selection['isochrone_model'])}, {selection['match_filter']}")

In [ ]:
stars = get_euclid_stars(CATALOG, cuts=default_star_cuts(CATALOG, "morphology_cuts"), subset=SUBSET)
footprint = survey_footprint_healpix_mask(stars["ra"].to_numpy(), stars["dec"].to_numpy(), NSIDE, nest=False)
# Inside the footprint, away from its edges (where the density is divided by the smoothed footprint)
interior = hp.smoothing((~footprint).astype(float), sigma=np.radians(SMOOTHING_DEG)) > 0.99
print(f"{len(stars):,} stars; footprint {(~footprint).sum() * hp.nside2pixarea(NSIDE, degrees=True):,.0f} deg2, "
      f"{interior.sum():,} interior pixels")

searches = {name: prepare_generic_search(stars, match_filter=selection["match_filter"],
                                         isochrone_model=selection["isochrone_model"], nside=NSIDE,
                                         footprint_mask=footprint, nest=False)
            for name, selection in SELECTIONS.items()}
for name, search in searches.items():
    print(f"{name}: {search.keep.sum():,} stars pass the cuts; isochrone {search.isochrones[0]}")

## 2. Maps at a few distance moduli

One row per selection: the residual (stars per pixel, its own colour scale, as the gifs of the search) and the
significance (the same scale for both selections, `SIGNIFICANCE_LIMITS`).

In [ ]:
results = {name: {dm: generic_search_single_dm(distance_modulus=dm, search=search, smoothing_deg=SMOOTHING_DEG)
                  for dm in DISTANCE_MODULI}
           for name, search in searches.items()}
significances = {name: {dm: generic_search_significance(result) for dm, result in by_dm.items()}
                 for name, by_dm in results.items()}

In [ ]:
for dm in DISTANCE_MODULI:
    fig, axes = plt.subplots(len(SELECTIONS), 2, figsize=(16.0, 5.5 * len(SELECTIONS)), layout="constrained")
    for row, name in zip(axes, SELECTIONS):
        result = results[name][dm]
        plot_generic_search_residual_map(result, ax=row[0], title=f"{name}: residual (stars per pixel)")
        plot_generic_search_residual_map({**result, "residual": significances[name][dm]}, ax=row[1],
                                         cmap="RdBu_r", vmin=SIGNIFICANCE_LIMITS[0], vmax=SIGNIFICANCE_LIMITS[1],
                                         colorbar_label="significance", title=f"{name}: significance")
    fig.suptitle(f"dm {dm:g}")
    plt.show()

## 3. Noise of the maps

Per selection and distance modulus, on the interior pixels: the stars selected, the background (stars per pixel), the
spread of the residual and of the significance (1.4826 × the median absolute deviation: robust to the streams and
other structures), and the fraction of pixels above 3 in significance (0.13% for Gaussian noise).

In [ ]:
def robust_std(values):
    """1.4826 x the median absolute deviation: the standard deviation of Gaussian values, robust to outliers."""
    return 1.4826 * np.median(np.abs(values - np.median(values)))


rows = []
for name, by_dm in results.items():
    for dm, result in by_dm.items():
        significance = significances[name][dm]
        usable = interior & ~np.ma.getmaskarray(significance)
        rows.append({
            "selection": name, "dm": dm, "n_selected": result["n_selected"],
            "background per pixel": float(np.ma.median(result["background"][usable])),
            "residual spread": robust_std(np.ma.getdata(result["residual"])[usable]),
            "significance spread": robust_std(np.ma.getdata(significance)[usable]),
            "fraction above 3": float(np.mean(np.ma.getdata(significance)[usable] > 3.0)),
        })
noise = pd.DataFrame(rows)
noise.round(3)

In [ ]:
# Distribution of the significance on the interior pixels, against a Gaussian of standard deviation 1
fig, axes = plt.subplots(1, len(DISTANCE_MODULI), figsize=(4.5 * len(DISTANCE_MODULI), 3.5), sharey=True,
                         layout="constrained")
bins = np.linspace(-6.0, 6.0, 61)
for ax, dm in zip(np.atleast_1d(axes), DISTANCE_MODULI):
    for name in SELECTIONS:
        significance = significances[name][dm]
        values = np.ma.getdata(significance)[interior & ~np.ma.getmaskarray(significance)]
        ax.hist(values, bins=bins, density=True, histtype="step", label=name)
    ax.plot(bins, np.exp(-bins**2 / 2) / np.sqrt(2 * np.pi), color="0.5", ls="--", label="Gaussian, std 1")
    ax.set(xlabel="significance", yscale="log", ylim=(1e-5, 1.0), title=f"dm {dm:g}")
np.atleast_1d(axes)[0].set_ylabel("density of pixels")
np.atleast_1d(axes)[-1].legend(fontsize="small")
plt.show()